# Conformal factor iteration (brute-force)

This notebook demonstrates the iteration from the pasted algorithm, using a brute-force $O(N^2)$ particle-particle summation in place of FMM.

Outputs:
- Convergence diagnostics
- A 1D slice of $\psi$ along the $x$-axis

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Convergence studies need full precision: print the minimal number of
# digits that uniquely round-trips each float64 (up to 17 significant).
np.set_printoptions(precision=17, floatmode='unique')

# Robustly add the project root to sys.path so we can import from ./src
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_root = None
for p in candidates:
    if (p / 'src' / 'conformal_solver.py').exists():
        project_root = p
        break
if project_root is None:
    raise FileNotFoundError("Could not find src/conformal_solver.py from current working directory")
sys.path.insert(0, str(project_root))

from src.conformal_solver import ConformalFactorSolver  # noqa: E402

In [ ]:
# Step 1: create a discrete cloud of interpolation points.
# For this demo we use a small uniform Cartesian grid in a box.
L = 2.0
n = 15  # keep small; brute-force is O(N^2)

# Construct the grid with the same floating-point operations as the C demo
# (dx = 2L/(n-1), x_i = -L + dx*i, vol = dx*dx*dx) rather than via
# np.linspace / dx**3, so both codes see bit-identical inputs.
dx = (2.0 * L) / (n - 1)
xs = -L + dx * np.arange(n)
X, Y, Z = np.meshgrid(xs, xs, xs, indexing='ij')
r = np.column_stack([X.ravel(), Y.ravel(), Z.ravel()])

volumes = np.full(r.shape[0], dx * dx * dx)

# Define a toy potential V(r).
# Use a smooth compact-ish bump via a Gaussian truncated by the box.
sigma = 0.5
r2 = np.sum(r*r, axis=1)
inv2sig2 = 1.0 / (2.0 * sigma * sigma)  # C multiplies by the reciprocal
V = np.exp(-r2 * inv2sig2)

# Optional: emulate 'support where V != 0' by dropping tiny values
eps = 1e-6
mask = V > eps
r_c = r[mask]
V_c = V[mask]
vol_c = volumes[mask]

r.shape, r_c.shape

In [ ]:
# Run the iteration (steps 4-11).
# M here is the parameter in the paper: M_ADM = 2M, ψ → 1 + M/r at infinity.
M = 1.0
solver = ConformalFactorSolver(r_c, V_c, vol_c, M=M, softening=0.5*dx, exclude_self=True)

psi, info = solver.solve(tol=1e-11, max_iter=200, error_norm='linf', verbose=True)
chi = psi - 1.0  # mass-normalized chi; use as input to psi_at_points

# Verify the mass condition: sum(m) should equal -M
m = solver.masses_from_chi(chi)
print(f"\nW          = {solver.W:.16e}")
print(f"sum(m)     = {m.sum():.16e}   (should be -M = {-M:.16e})")
print(f"last_error = {info.last_error:.16e}")

# Physical psi at grid points via step 12 (brute-force summation of converged masses)
psi_grid = solver.psi_at_points(r_c, chi=chi)
print(f"psi (step 12) at grid points: min={psi_grid.min():.16e}  max={psi_grid.max():.16e}")

info

In [ ]:
# Step 12: evaluate psi along the x-axis at y=z=0 using brute-force summation.
# ψ(r) = 1 − Σ_a m_a / |r − r_a|  (1/(4π) absorbed into masses)
K = 200
# Same arithmetic as the C demo: x_i = xmin + (xmax - xmin)*i/(K - 1)
xmin, xmax = -3.0, 3.0
xline = xmin + (xmax - xmin) * np.arange(K) / (K - 1)
r_eval = np.column_stack([xline, np.zeros_like(xline), np.zeros_like(xline)])
psi_line = solver.psi_at_points(r_eval, chi=chi)

plt.figure(figsize=(7, 4))
plt.plot(xline, psi_line)
plt.axhline(1.0, color='gray', ls='--', lw=0.8)
plt.xlabel('x')
plt.ylabel(r'$\psi(x,0,0)$')
plt.title(r'Conformal factor along x-axis (brute-force, $M_{ADM}=2M=$' + f'{2*M})')
plt.grid(True)
plt.show()

In [ ]:
# Full-precision output for convergence studies.
# The format is identical to the C demo's (%23.16e = 17 significant digits, the
# round-trip precision of a float64), so the .dat files can be differenced directly.
FMT = '%23.16e'
zeros = np.zeros_like(xline)

np.savetxt(
    'psi_grid_python.dat',
    np.column_stack([r_c, psi_grid]),
    fmt=FMT,
    header='1:x 2:y 3:z 4:psi',
)
np.savetxt(
    'psi_xcut_python.dat',
    np.column_stack([xline, zeros, zeros, psi_line]),
    fmt=FMT,
    header=f'1:x 2:y 3:z 4:psi  (x-axis cut via psi_at_points, K={K})',
)

print(f"Nkeep      = {r_c.shape[0]}")
print(f"dx         = {dx:.16e}")
print(f"W          = {solver.W:.16e}")
print(f"psi_min    = {psi_grid.min():.16e}")
print(f"psi_max    = {psi_grid.max():.16e}")
print(f"last_error = {info.last_error:.16e}")
print("wrote psi_grid_python.dat, psi_xcut_python.dat")
